In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import os

os.environ['TF_CUDNN_DETERMINISTIC']='1'
# os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'

import tensorflow as tf


SEED = 1
def set_seed(seed=SEED):
    tf.keras.utils.set_random_seed(seed)
set_seed()

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory


# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Loading the dataset

In [ ]:
data = pd.read_csv(
    '/kaggle/input/fashion-product-images-small/styles.csv',
    on_bad_lines = 'skip'  
                  )

data = data.dropna()
data = pd.DataFrame(data)

image_path = '/kaggle/input/fashion-product-images-small/images'

In [ ]:
data

# Dataset visualization

In [ ]:
def show_k_random(k, name):
    choices = np.random.choice(data.index, k, replace=False)
    choices = data.loc[choices, ['id', name]]
    fig, axs = plt.subplots(1, k, figsize = (15, 10))
    for this_name, id, ax in zip(choices[name], choices['id'], axs.ravel()):
        img_path = f"{image_path}/{id}.jpg"
        img = plt.imread(img_path)
        ax.imshow(img, cmap='gray')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f'{this_name}')
    plt.tight_layout()
    plt.show()

In [ ]:
labels = data.columns[1:-1]
for col in labels:
    print(col)
    show_k_random(10, col)

# Extracting labels

In [ ]:
def get_lookup(name):
    return tf.keras.layers.StringLookup(
        vocabulary=(data[name]).astype(str).unique(),
        oov_token='<OOV>',
        dtype=tf.int64
    )

label_lookups = {}
for label in labels:
    label_lookups[label] = get_lookup(label)
    
def get_labels(row):
    label_list = {}
    label_list['id'] = row['id']
    label_list['path'] = path = f"{image_path}/{row['id']}.jpg"
    for label in labels:
        label_list[label] = int(label_lookups[label](str(row[label])))
    return label_list

In [ ]:
from tqdm.notebook import tqdm

new_data = []
if os.path.exists('/kaggle/input/data-labels-fashion-product-images-small/data_labels.csv'):
    print('Loading...')
    new_data = pd.read_csv('/kaggle/input/data-labels-fashion-product-images-small/data_labels.csv')
else:
    for i, row in tqdm(data.iterrows(), total=data.shape[0]):
        path = f"{image_path}/{row['id']}.jpg"
        if os.path.exists(path):
            new_data.append(get_labels(row))
        
print(f'Successfully loaded images: {len(new_data)} ({len(new_data) / len(data) * 100}%)')

In [ ]:
data_labels = pd.DataFrame(new_data)
data_labels.to_csv('data_labels.csv', index=False)
data_labels

# Splitting the data

In [ ]:
# Hyperparams
BATCH_SIZE = 64
IMAGE_SIZE = [80, 60]

In [ ]:
from sklearn.model_selection import train_test_split

# Creating the TensorFlow dataset
# Helper Class
class splitter:
    def __init__(self, category, df, num_classes,
                image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
                train_size=0.6, valid_size=0.3, test_size=0.1, silent=False
                ):
        assert (train_size + test_size + valid_size == 1), "Test + Train + Valid must be 1"
        
        self.category = category
        self.image_size = image_size
        self.batch_size = batch_size
        self.num_classes = num_classes
        test, train, valid = self.split(df, train_size, test_size, valid_size, silent)
        self.test_ds = self.create_tf_dataset(test)
        self.train_ds = self.create_tf_dataset(train)
        self.valid_ds = self.create_tf_dataset(valid)
    
    def split(self, df, train_size, test_size, valid_size, silent):
        # Split data into training and test set
        train_df, temp_df = train_test_split(df, test_size=(test_size+valid_size), random_state=SEED)

        # Split the temporary set into validation and test sets
        validation_df, test_df = train_test_split(temp_df, test_size=(test_size/(valid_size + test_size)), random_state=SEED)

        if not silent:
            # Print the sizes of each set
            print(self.category + ":")
            print(f"\tTraining set size: {len(train_df)} ({int(len(train_df)/len(data) * 100)}%)")
            print(f"\tValidation set size: {len(validation_df)} ({int(len(validation_df)/len(data) * 100)}%)")
            print(f"\tTest set size: {len(test_df)} ({int(len(test_df)/len(data) * 100)}%)")
        
        return test_df, train_df, validation_df
    
    def load_image(self, image_path):
        image = tf.io.read_file(image_path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.cast(image, tf.float32)
        image = tf.image.resize(image, self.image_size)  # Resize to the desired size
        image = image / 255.0  # Normalize to [0, 1]
        return image

    def map_func(self, path, label):
        image = self.load_image(path)
        return image, label

    def create_tf_dataset(self, df):
        x = df['path'].tolist()
        y = np.array(df[self.category].tolist(), dtype=np.int32)
        y -= 1
        y_one_hot = tf.keras.utils.to_categorical(y, num_classes=self.num_classes)
        
        dataset = tf.data.Dataset.from_tensor_slices((x, y_one_hot))
        dataset = dataset.map(lambda path, label: self.map_func(path, label))
        dataset = dataset.shuffle(buffer_size=len(df))
        dataset = dataset.batch(self.batch_size)
        dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
        return dataset
    
    def get_data(self):
        return self.train_ds, self.valid_ds, self.test_ds, self.num_classes

In [ ]:
splitters = {}
for label in labels:
    num_classes = int(data_labels[label].max())
    splitters[label] = splitter(label, data_labels, num_classes)

# Neural Network Model

In [ ]:
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler


# A class to help train and analyze
class NNTrainer:
    def __init__(self, model: keras.Sequential, test_ds, train_ds, valid_ds, category, epochs):
        self.model = model
        self.test_ds = test_ds
        self.train_ds = train_ds
        self.valid_ds = valid_ds
        self.category = category
        self.epochs = epochs
        self.history_frame = None
        
    def train(self, callbacks=None):
        if callbacks == None:
            early_stopping = EarlyStopping(
                monitor='val_loss',
                patience=10,
                restore_best_weights=True,
            )

            lr_scheduler = LearningRateScheduler(lambda epoch, lr: lr if epoch < 10 else lr * 0.99)
            callbacks = [early_stopping, lr_scheduler]
                
        print('Training with:')
        for c in callbacks:
            print(str(type(c)).split('.')[-1][:-2])
        self.history = self.model.fit(
            self.train_ds,
            validation_data=self.valid_ds,
            epochs=self.epochs,
            callbacks=callbacks
        )
        
        if self.history_frame == None:
            self.history_frame = pd.DataFrame(self.history.history)
        
    def plot(self):
        # Create a figure and axes
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        # Plot loss
        axes[0].plot(self.history_frame.index, self.history_frame['loss'], label='Training Loss')
        axes[0].plot(self.history_frame.index, self.history_frame['val_loss'], label='Validation Loss')
        axes[0].set_title('Loss')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].legend()

        # Plot accuracy
        axes[1].plot(self.history_frame.index, self.history_frame['categorical_accuracy'], label='Training Accuracy')
        axes[1].plot(self.history_frame.index, self.history_frame['val_categorical_accuracy'], label='Validation Accuracy')
        axes[1].set_title('Accuracy')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Accuracy')
        axes[1].legend()
        
    def test(self, show=False, silent=False):
        test_loss, test_accuracy = self.model.evaluate(self.test_ds, verbose=int(not show and not silent))
        if show and not silent:
            print(f'{self.category}:\tTest loss: {test_loss}, Test accuracy: {test_accuracy}')
        return test_loss, test_accuracy
    
    def summary(self):
        self.model.summary()

In [ ]:
from tensorflow.keras import layers

def get_model(num_classes, category, model_name='init', replace=False):
    if (not replace) and os.path.exists(f"{category}_model_{model_name}.keras") and \
        os.path.exists(f"{category}_history_{model_name}.csv"):
        print(f'{category}_model_{model_name} exists...')
        model = keras.models.load_model(f"{category}_model_{model_name}.keras")
        return model
    
    
    print(f'{category} model does not exist')
    model = keras.Sequential([
        layers.Input(shape=(*IMAGE_SIZE, 3)),

        # Block 1
        layers.Conv2D(filters=max(8, num_classes // 16), kernel_size=(3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),

        # Block 2
        layers.Conv2D(filters=max(16, num_classes // 8), kernel_size=(3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),

        # Flatten and dense layers
        layers.Flatten(),
        layers.Dense(max(64, num_classes // 2), activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ], name=f'{category}_model')

    model.compile(
        optimizer='adam',
        loss=tf.keras.losses.CategoricalCrossentropy(),  # For one-hot encoded labels
        metrics=[tf.keras.metrics.CategoricalAccuracy()]
    )
    
    return model

In [ ]:
def get_nnt(df, category, epochs, model_name='init', model=None, replace=False):
    train, valid, test, num_classes = splitters[category].get_data()
    num_classes = int(df[category].max())
    if model == None:
        model = get_model(num_classes, category, model_name, replace)
    nnt = NNTrainer(model, test, train, valid, category, epochs)
    if not replace:
        p_m = f"{category}_model_{model_name}.keras"
        p_h = f"{nnt.category}_history_{model_name}.csv"
        print(p_m)
        if os.path.exists(p_h) and os.path.exists(p_m):
            print('Loading history...')
            nnt.history_frame = pd.read_csv(p_h)
            new_model = keras.models.load_model(p_m)
            nnt.model = new_model
    return nnt

def get_nnts(df, labels, epochs=50, model_name='init'):
    nnts = {}
    for label in labels:
        nnts[label] = get_nnt(df, label, epochs, model_name)
    return nnts

def train_one(nnt, model_name='init', callbacks=None, replace=False):
    p_m = f"{nnt.category}_model_{model_name}.keras"
    p_h = f"{nnt.category}_history_{model_name}.csv"
    if (not replace) and os.path.exists(p_m) and os.path.exists(p_h):
        print('Model exists, training aborted.')
    else:
        nnt.train(callbacks)
        nnt.model.save(p_m)
        nnt.history_frame.to_csv(p_h)
        
def train_all(nnts, model_name='init'):
    for nnt in nnts:
        print(f'Training {nnt}...')
        train_one(nnts[nnt], model_name)

In [ ]:
nnts = get_nnts(data_labels, labels)
for nnt in nnts:
    nnts[nnt].summary()

In [ ]:
train_all(nnts)

## Gender

In [ ]:
category = 'gender'
nnt = nnts[category]

In [ ]:
nnt.plot()

The plot shows a possible overfitting model, we can add a LR controller on plateau to avoid it.

In [ ]:
num_classes = splitters[category].num_classes
num_classes

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

n_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0)
]

In [ ]:
nnt_new_gender = get_nnt(data_labels, category, epochs=50, model_name='enhanced')
train_one(nnt_new_gender, model_name='enhanced', callbacks=n_callbacks)

In [ ]:
nnt_new_gender.plot()

In [ ]:
nnt_new_gender.test(show=True)

In [ ]:
nnt.test(show=True)

The results are not so bad, but let's see if using a pretrained model can improve it.

In [ ]:
IMAGE_SIZE_MBN = [96, 96]
splitters[category] = splitter(category, data_labels, num_classes, image_size=IMAGE_SIZE_MBN)

In [ ]:
from tensorflow.keras import applications, models

def get_mobilenet_based_model(category, num_classes, fine_tune_at=100):
    base_model_mobilenet = applications.MobileNetV2(
        input_shape=(*IMAGE_SIZE_MBN, 3),
        include_top=False,
        weights='imagenet'
    )

    base_model_mobilenet.trainable = True

    for layer in base_model_mobilenet.layers[:fine_tune_at]:
        layer.trainable = False

    inputs = layers.Input(shape=(*IMAGE_SIZE_MBN, 3))
    x = base_model_mobilenet(inputs)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name=f'{category}_mbn')

    model.compile(
        optimizer='adam',
        loss=tf.keras.losses.CategoricalCrossentropy(),  # For one-hot encoded labels
        metrics=[tf.keras.metrics.CategoricalAccuracy()]
    )
    
    return model

In [ ]:
model = get_mobilenet_based_model(category, num_classes)
model.summary()

In [ ]:
nnt_new_gender = get_nnt(data_labels, category, epochs=50, model_name='mobile_net', model=model)
train_one(nnt_new_gender, model_name='mobile_net')

In [ ]:
nnt_new_gender.plot()
nnt_new_gender.test(show=True)

So, the best we can achieve is about 90%.

## Master Category

In [ ]:
category = labels[1]
nnt = get_nnt(data_labels, category, 50)

In [ ]:
nnt.plot()

In [ ]:
nnt.test(show=True)

Therefore, the test is okay and the plots are fine (Sudden decrease in accuracy is probably due to lack of data but the decrease is not dramatic).

## Sub Category

In [ ]:
category = labels[2]
nnt = get_nnt(data_labels, category, 50)

In [ ]:
nnt.plot()

In [ ]:
nnt.test(show=True)

In [ ]:
splitters[category].num_classes

We might need to make early stopping faster and reduce LR.

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=6,
        restore_best_weights=True,
    ),
    ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.2, 
        patience=3, 
        min_lr=0.0001
    )]

nnt_new_subcategory = get_nnt(data_labels, category, epochs=50, model_name='enhanced')
train_one(nnt_new_subcategory, callbacks=callbacks, model_name='enhanced')

In [ ]:
nnt_new_subcategory.plot()

In [ ]:
nnt_new_subcategory.test(show=True)

The plot is more stable and it's not overfitting now.

## Article Type

In [ ]:
category = labels[3]
nnt = get_nnt(data_labels, category, 50)

In [ ]:
nnt.plot()

In [ ]:
splitters[category].num_classes

Since the number of classes is too high, we can use data augmentation and bigger batch size to avoid this.

In [ ]:
# Batch size
num_classes = int(data_labels[category].max())
splitters[category] = splitter(category, data_labels, num_classes, batch_size=128)

# Data augmentation

model = keras.Sequential([
    layers.Input(shape=(*IMAGE_SIZE, 3)),
    layers.RandomFlip(seed=SEED),
    layers.RandomZoom(height_factor=(-0.2, 0.2), seed=SEED),

    # Block 1
    layers.Conv2D(filters=max(8, num_classes // 16), kernel_size=(3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # Block 2
    layers.Conv2D(filters=max(16, num_classes // 8), kernel_size=(3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # Flatten and dense layers
    layers.Flatten(),
    layers.Dense(max(64, num_classes // 2), activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
], name=f'{category}_model')

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(),  # For one-hot encoded labels
    metrics=[tf.keras.metrics.CategoricalAccuracy()]
)

model.summary()

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=6,
        restore_best_weights=True,
    ),
    ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.2, 
        patience=3, 
        min_lr=0.0001
    )]

nnt_new_articletype = get_nnt(data_labels, category, epochs=50, model_name='enhanced', model=model)
train_one(nnt_new_articletype, callbacks=callbacks, model_name='enhanced')

In [ ]:
nnt_new_articletype.plot()

Val loss is less than training loss because of the augmentation. Also, the curve is less noisy!

In [ ]:
nnt_new_articletype.test(show=True)

We can try MobileNet to see how it can improve performance.

In [ ]:
model = get_mobilenet_based_model(category, num_classes)
model.summary()

In [ ]:
splitters[category] = splitter(category, data_labels, num_classes, image_size=IMAGE_SIZE_MBN)

In [ ]:
nnt_new_articletype = get_nnt(data_labels, category, epochs=50, model_name='mobile_net', model=model)

In [ ]:
nnt_new_articletype.model.layers

In [ ]:
train_one(nnt_new_articletype, model_name='mobile_net')

In [ ]:
nnt_new_articletype.plot()

In [ ]:
nnt_new_articletype.test(show=True)

Although the curve is noisier, the accuracy improved, so this model is better and due to vast number of classes, the results are quite fine.

## Base Colour

In [ ]:
category = labels[4]
nnt = get_nnt(data_labels, category, 50)

In [ ]:
nnt.plot()

In [ ]:
nnt.test(show=True)

Low test accuracy and the spike in learning curve indicate a model that is not so good. We can feed the colour histogram into the NN instead.

In [ ]:
def show_histograms(image, hue_hist, saturation_hist, value_hist):
    """
    Displays the image and its color histograms (Hue, Saturation, and Value) using a mosaic layout.
    
    Parameters:
    - image (tensorflow.Tensor): The image to display.
    - hue_hist (tensorflow.Tensor): Histogram for the Hue channel.
    - saturation_hist (tensorflow.Tensor): Histogram for the Saturation channel.
    - value_hist (tensorflow.Tensor): Histogram for the Value channel.
    """
    # Convert image from TensorFlow tensor to numpy array
    image_np = image.numpy()
    
    # Convert from RGB to HSV color space
    image_hsv = tf.image.rgb_to_hsv(image_np)
    
    # Define the mosaic layout
    fig, axs = plt.subplot_mosaic([
        ['image', 'hue'],
        ['image', 'value'],
        ['image', 'saturation']
    ], layout='constrained', figsize=(12, 6))

    # Plot the image
    axs['image'].imshow(image_np)
    axs['image'].set_title('Image')
    axs['image'].axis('off')

    # Plot Hue histogram with logarithmic scale
    axs['hue'].plot(tf.math.log1p(hue_hist).numpy(), color='r')
    axs['hue'].set_title('Hue Histogram')
    axs['hue'].set_ylim(0, tf.reduce_max(tf.math.log1p(hue_hist).numpy()) * 1.1)

    # Plot Saturation histogram with logarithmic scale
    axs['saturation'].plot(tf.math.log1p(saturation_hist).numpy(), color='g')
    axs['saturation'].set_title('Saturation Histogram')
    axs['saturation'].set_ylim(0, tf.reduce_max(tf.math.log1p(saturation_hist).numpy()) * 1.1)

    # Plot Value histogram with logarithmic scale
    axs['value'].plot(tf.math.log1p(value_hist).numpy(), color='b')
    axs['value'].set_title('Value Histogram')
    axs['value'].set_ylim(0, tf.reduce_max(tf.math.log1p(value_hist).numpy()) * 1.1)

    plt.show()


def extract_color_histograms_from_tensor(image_tensor, bins=256, show=False):
    # Convert TensorFlow tensor to HSV color space
    image_hsv = tf.image.rgb_to_hsv(image_tensor)

    # Define a threshold to identify white pixels (background)
    threshold = 0.9 
    
    # Create a mask for non-background pixels
    background_mask = tf.reduce_all(image_tensor > threshold, axis=-1)

    # Use the mask to filter out background pixels
    foreground_mask = tf.logical_not(background_mask)

    # Apply the mask to the HSV image
    hue_foreground = tf.boolean_mask(image_hsv[:, :, 0], foreground_mask)
    saturation_foreground = tf.boolean_mask(image_hsv[:, :, 1], foreground_mask)
    value_foreground = tf.boolean_mask(image_hsv[:, :, 2], foreground_mask)

    # Calculate histograms for each channel
    hue_hist = tf.histogram_fixed_width(hue_foreground, [0, 1], nbins=bins)
    saturation_hist = tf.histogram_fixed_width(saturation_foreground, [0, 1], nbins=bins)
    value_hist = tf.histogram_fixed_width(value_foreground, [0, 1], nbins=bins)

    # Flatten and normalize histograms
    hue_hist = tf.divide(hue_hist, tf.reduce_sum(hue_hist))
    saturation_hist = tf.divide(saturation_hist, tf.reduce_sum(saturation_hist))
    value_hist = tf.divide(value_hist, tf.reduce_sum(value_hist))

    if show:
        show_histograms(image_tensor, hue_hist, saturation_hist, value_hist)

    # Concatenate histograms into a single array
    histograms = tf.stack([hue_hist, saturation_hist, value_hist], axis=-1)

    return histograms


    
def extract_color_histograms_from_path(image_path, bins=256, show=False, image_size=IMAGE_SIZE):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(image, channels=3)  # Decode as RGB image
    image = tf.image.resize(image, image_size)  # Resize if needed
    image = tf.cast(image, tf.float32) / 255.0  # Normalize to [0, 1]

    return extract_color_histograms_from_tensor(image, bins, show)

def get_histogram_size(bins=256, channels=3):
    return (bins, channels)

In [ ]:
# Example usage
path = data_labels.iloc[0]['path']
histograms = extract_color_histograms_from_path(path, show=True)
print(histograms.shape)

In [ ]:
label_lookups[category].get_vocabulary()[data_labels.iloc[0]['baseColour']]

The hue histogram well-defines the colour!

In [ ]:
class splitter_with_histogram:
    def __init__(self, category, df, num_classes,
                image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
                train_size=0.6, valid_size=0.3, test_size=0.1, silent=False
                ):
        assert (train_size + test_size + valid_size == 1), "Test + Train + Valid must be 1"
        
        self.category = category
        self.image_size = image_size
        self.hist_size = get_histogram_size()
        self.batch_size = batch_size
        self.num_classes = num_classes
        test, train, valid = self.split(df, train_size, test_size, valid_size, silent)
        self.test_ds = self.create_tf_dataset(test)
        self.train_ds = self.create_tf_dataset(train)
        self.valid_ds = self.create_tf_dataset(valid)
    
    def split(self, df, train_size, test_size, valid_size, silent):
        # Split data into training and test set
        train_df, temp_df = train_test_split(df, test_size=(test_size+valid_size), random_state=SEED)
        # Split the temporary set into validation and test sets
        validation_df, test_df = train_test_split(temp_df, test_size=(test_size/(valid_size + test_size)), random_state=SEED)
        
        if not silent:
            # Print the sizes of each set
            print(self.category + ":")
            print(f"\tTraining set size: {len(train_df)} ({int(len(train_df)/len(data) * 100)}%)")
            print(f"\tValidation set size: {len(validation_df)} ({int(len(validation_df)/len(data) * 100)}%)")
            print(f"\tTest set size: {len(test_df)} ({int(len(test_df)/len(data) * 100)}%)")
        
        return test_df, train_df, validation_df
    
    def load_image_with_histogram(self, image_path):
        image = tf.io.read_file(image_path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.cast(image, tf.float32)
        image = tf.image.resize(image, self.image_size)  # Resize to the desired size
        image = image / 255.0  # Normalize to [0, 1]
        return extract_color_histograms_from_tensor(image)

    def map_func(self, path, label):
        histogram = self.load_image_with_histogram(path)
        return histogram, label

    def create_tf_dataset(self, df):
        x = df['path'].tolist()
        y = np.array(df[self.category].tolist(), dtype=np.int32)
        y -= 1
        y_one_hot = tf.keras.utils.to_categorical(y, num_classes=self.num_classes)
        
        dataset = tf.data.Dataset.from_tensor_slices((x, y_one_hot))
        dataset = dataset.map(lambda path, label: self.map_func(path, label))
        dataset = dataset.shuffle(buffer_size=len(df))
        dataset = dataset.batch(self.batch_size)
        dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
        return dataset
    
    def get_data(self):
        return self.train_ds, self.valid_ds, self.test_ds, self.num_classes

In [ ]:
num_classes = len(label_lookups[category].get_vocabulary()) - 1

In [ ]:
splitters[category] = splitter_with_histogram(category,
                             data_labels,
                             num_classes)

In [ ]:
image_size = splitters[category].hist_size
image_size

In [ ]:
model = keras.Sequential([
    layers.Input(shape=image_size),
    layers.Conv1D(64, kernel_size=5, activation='relu', padding='same'),  
    layers.MaxPooling1D(pool_size=2),
    layers.BatchNormalization(),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),  # Increased units
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')  # Output layer
], name=f'{category}_hist')

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(),  # For one-hot encoded labels
    metrics=[tf.keras.metrics.CategoricalAccuracy()]
)

In [ ]:
nnt_new_basecolour = get_nnt(data_labels, category, epochs=50, model_name='hist', model=model)
nnt_new_basecolour.summary()

In [ ]:
train_one(nnt_new_basecolour, model_name='hist')

In [ ]:
nnt_new_basecolour.plot()

In [ ]:
nnt_new_basecolour.test(show=True)

## Season

In [ ]:
category = labels[5]
nnt = get_nnt(data_labels, category, 50)

In [ ]:
nnt.plot()

In [ ]:
nnt.test(show=True)

The result indicate an overfitting model. Let's try some callbacks...

In [ ]:
num_classes = splitters[category].num_classes
num_classes

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

n_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0)
]

In [ ]:
nnt_new_season = get_nnt(data_labels, category, epochs=50, model_name='enhanced')
train_one(nnt_new_season, model_name='enhanced', callbacks=n_callbacks)

In [ ]:
nnt_new_season.plot()

In [ ]:
nnt_new_season.test(show=True)

The results are better by a small bit. Let's try mobilenet.

In [ ]:
model = get_mobilenet_based_model(category, num_classes, fine_tune_at=-1)
model.summary()

In [ ]:
splitters[category] = splitter(category, data_labels, num_classes, image_size=IMAGE_SIZE_MBN)

In [ ]:
nnt_new_season = get_nnt(data_labels, category, epochs=50, model_name='mobile_net', model=model)

In [ ]:
n_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0)
]

In [ ]:
train_one(nnt_new_season, callbacks=n_callbacks, model_name='mobile_net')

In [ ]:
nnt_new_season.plot()
nnt_new_season.test(show=True)

The results improved with mobilenet.

## Year

In [ ]:
category = labels[6]
nnt = get_nnt(data_labels, category, 50)

In [ ]:
nnt.plot()

In [ ]:
nnt.test(show=True)

The results are not so bad since the year is not quite related to the image content, but we can improve it using reducingLR.

In [ ]:
num_classes = splitters[category].num_classes
num_classes

n_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0)
]

In [ ]:
nnt_new_year = get_nnt(data_labels, category, epochs=50, model_name='enhanced')
train_one(nnt_new_year, model_name='enhanced', callbacks=n_callbacks)

In [ ]:
nnt_new_year.plot()
nnt_new_year.test(show=True)

The model's accuracy and stability improved.

## Usage

In [ ]:
category = labels[7]
nnt = get_nnt(data_labels, category, 50)

In [ ]:
nnt.plot()

In [ ]:
nnt.test(show=True)

The results are okay, but maybe some controlled LR will make the curve more stable.

In [ ]:
num_classes = splitters[category].num_classes

n_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0)
]

In [ ]:
nnt_new_usage = get_nnt(data_labels, category, epochs=50, model_name='enhanced')
train_one(nnt_new_usage, model_name='enhanced', callbacks=n_callbacks)

In [ ]:
nnt_new_usage.plot()
nnt_new_usage.test(show=True)

The curve is more stable but the accuracy did not improve so much. Nevertheless, this model is better due to its curve.

# Final Model

Now its time to combine all models into one. First, we have to detect which models are the best.

## Best models

In [ ]:
class ModelManager:
    def __init__(self, path='.', image_size_mbn=IMAGE_SIZE_MBN, image_size=IMAGE_SIZE):
        self.path = path
        self.image_size_mbn = image_size_mbn
        self.image_size = image_size
        self.models = {}
        
    def get_all_models(self):
        """
        Loads all models from the specified directory and stores them in the models dictionary.
        """
        pattern = re.compile(r'^(?P<category>.+)_model_(?P<type>.+)\.keras$')
        for filename in os.listdir(self.path):
            match = pattern.match(filename)
            if match:
                category = match.group('category')
                model_name = match.group('type')
                if category not in self.models:
                    self.models[category] = {}
                model = keras.models.load_model(os.path.join(self.path, filename))
                self.models[category][model_name] = {
                    'model': model, 
                    'input_shape': model.input_shape[1:], 
                    'num_classes': model.layers[-1].units
                }
        return self.models

    def test_all_models(self, df, hist='hist'):
        """
        Tests all models and stores the results in the models dictionary.
        """
        for category in tqdm(self.models):
            for model_name in self.models[category]:
                model = self.models[category][model_name]['model']
                input_shape = self.models[category][model_name]['input_shape']
                num_classes = self.models[category][model_name]['num_classes']
                if input_shape == (*self.image_size_mbn, 3):
                    splt = splitter(category, df, num_classes, image_size=self.image_size_mbn, silent=True)
                elif input_shape == (*self.image_size, 3):
                    splt = splitter(category, df, num_classes, image_size=self.image_size, silent=True)
                elif model_name == hist:
                    splt = splitter_with_histogram(category, df, num_classes, image_size=self.image_size, silent=True)
                else:
                    raise ValueError(f'input shape {input_shape} is not supported.')
                
                train, valid, test, _ = splt.get_data()
                nnt = NNTrainer(model, test, train, valid, category, epochs=0)
                
                p_m = f"{category}_model_{model_name}.keras"
                p_h = f"{nnt.category}_history_{model_name}.csv"
                p_m = os.path.join(self.path, p_m)
                p_h = os.path.join(self.path, p_h)
                if os.path.exists(p_h) and os.path.exists(p_m):
                    nnt.history_frame = pd.read_csv(p_h)
                    new_model = keras.models.load_model(p_m)
                    nnt.model = new_model
                
                self.models[category][model_name]['test_result'] = nnt.test(silent=True)[1]
        return self.models
    
    @staticmethod
    def color_value(is_max):
        """
        Colors the cell based on whether it is the maximum value in its row.
        """
        return 'background-color: gold; color: black' if is_max else ''
    
    @staticmethod
    def highlight_best_in_row(df):
        """
        Highlights the best value in each row.
        """
        best_models = []

        def apply_styles(row):
            is_max = row == row.max()
            best_model = row[is_max].index[0] if is_max.any() else None
            best_models.append((row.name, best_model))
            return [ModelManager.color_value(is_max_value) for is_max_value in is_max]
        
        styled_df = df.style.apply(apply_styles, axis=1)
        
        return styled_df, best_models

    def extract_attribute_table(self, attribute):
        """
        Extracts and styles a table with the specified attribute.
        """
        rows = []

        for category, model_dict in self.models.items():
            for model_name, attrs in model_dict.items():
                attr_value = f"{attrs.get(attribute, 0) * 100:.1f}%" if attrs.get(attribute) is not None else "Not Tested"
                rows.append((category, model_name, attr_value))

        df = pd.DataFrame(rows, columns=['Category', 'Model Name', attribute.replace('_', ' ').title()])
        pivot_df = df.pivot(index='Category', columns='Model Name', values=attribute.replace('_', ' ').title())
        pivot_df = pivot_df.fillna('-')
        styled_df, best_models = self.highlight_best_in_row(pivot_df)
        return styled_df, best_models

manager = ModelManager()

In [ ]:
manager.get_all_models()

In [ ]:
manager.test_all_models(data_labels)
styled_df, best_models = manager.extract_attribute_table('test_result')

In [ ]:
# Results
styled_df

In [ ]:
# Best Models
best_models

In [ ]:
models_keras = {}
for cat, typ in best_models:
    models_keras[cat] = {
        'model': manager.models[cat][typ]['model'], 
         'input_shape': manager.models[cat][typ]['input_shape']
    }
models_keras

In [ ]:
from tensorflow.keras.preprocessing import image

class ModelPredictor:
    def __init__(self, models_keras, label_lookups):
        """
        Initializes the ModelPredictor with a dictionary of models.
        """
        self.models = models_keras
        self.label_lookups = label_lookups
    
    def load_image(self, image_path, target_size):
        """
        Loads and preprocesses the image.
        """
        img = image.load_img(image_path, target_size=target_size)
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
#         img_array = preprocess_input(img_array)
        return img_array
    
    def predict(self, image_path):
        """
        Predicts the class for the given image path using the best model for each category.
        """
        predictions = {}
        for category in self.models:
            best_model = self.models[category]['model']
            input_shape = self.models[category]['input_shape']
            
            img_array = self.load_image(image_path, target_size=input_shape[:2])
            preds = best_model.predict(img_array, verbose=0)
            
            predicted_class = np.argmax(preds, axis=1)[0]
            predictions[category] = label_lookups[category].get_vocabulary()[predicted_class+1]
            
        return predictions
    
predictor = ModelPredictor(models_keras, label_lookups)

In [ ]:
def pred_values(row, df, predictor, label_lookups):
    preds = predictor.predict(df.iloc[row]['path'])
    reals = {}
    for cat in predictor.models:
        real_class = df.iloc[row][cat]
        reals[cat] = label_lookups[cat].get_vocabulary()[real_class]
    return reals, preds

In [ ]:
def plot_comparison_with_image(real_values, preds, image_path):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    
    # Table Display
    ax1.axis('off')  # Hide the axes for the table display
    
    # Prepare table data
    table_data = []
    for key in real_values:
        real_value = real_values[key]
        pred_value = preds.get(key, 'Not Predicted')
        table_data.append([key, pred_value, real_value])
    
    # Create the table
    table = ax1.table(cellText=table_data,
                      colLabels=['Category', 'Predicted', 'Real'],
                      cellLoc='center', 
                      loc='center', 
                      bbox=[0, 0, 1, 1],
                      edges='closed')  # Add borders around the cells
    
    # Style the table
    table.auto_set_font_size(False)
    table.set_fontsize(18)
    table.auto_set_column_width([0, 1, 2])
    
    # Improve table aesthetics
    for (i, j), cell in table._cells.items():
        cell.set_edgecolor('black')
        cell.set_linewidth(1)
        if i == 0:  # Header row
            cell.set_text_props(weight='bold', color='white')
            cell.set_facecolor('gray')
        else:
            cell.set_facecolor('lightgray' if i % 2 == 0 else 'white')
    
    # Image Display
    img = plt.imread(image_path)
    ax2.imshow(img)
    ax2.axis('off')  # Hide axes for image
    ax2.set_title('Image')
    
    plt.tight_layout()
    plt.show()
    
def plot_predictions(row, df, predictor, label_lookups):
    reals, preds = pred_values(row, df, predictor, label_lookups)
    path = df.iloc[row]['path']
    plot_comparison_with_image(reals, preds, path)

In [ ]:
plot_predictions(12000, data_labels, predictor, label_lookups)

# Checking Data Distribution

In [ ]:
data_labels.hist(figsize=(16, 12), grid=False, edgecolor='black', linewidth=1.2)

In [ ]:
# Calculate class weights for each label
from sklearn.utils.class_weight import compute_class_weight

class_weights = {}
for label in labels:
    class_weights[label] = compute_class_weight(
        'balanced',
        classes=np.unique(data_labels[label]),
        y=data_labels[label]
    )

# New Neural Network Model

In [ ]:
from keras.applications.resnet import preprocess_input
import cv2

IMAGE_DATA_PATH = './image_data.npy'

def load_image(imagePath, image_size=IMAGE_SIZE):
    image = cv2.imread(imagePath)
    image = cv2.resize(image, (image_size[1], image_size[0]))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = preprocess_input(image)
    return image

if not os.path.exists(IMAGE_DATA_PATH):
    image_data = []
    for path in tqdm(data_labels['path']):
        image_data.append(load_image(path))
    image_data = np.array(image_data, dtype="float")
    np.save(IMAGE_DATA_PATH, image_data)
else:
    image_data = np.load(IMAGE_DATA_PATH)

In [ ]:
from sklearn.preprocessing import LabelBinarizer

y_binarized = []
LB = {}
for label in labels:
    LB[label] = LabelBinarizer()
    y_binarized.append(LB[label].fit_transform(np.array(data_labels[label])))

In [ ]:
split = train_test_split(image_data, *y_binarized, test_size=0.3, random_state=SEED)

In [ ]:
X_train, X_test = split[0], split[1]
X_val, X_test = train_test_split(X_test, test_size=(0.1/0.3), random_state=SEED)
y_train = {}
y_val = {}
y_test = {}
for i in range(2, len(split), 2):
    name = labels[i // 2 - 1]
    x, y = split[i], split[i+1]
    y, z = train_test_split(y, test_size=(0.1/0.3) ,random_state=SEED)
    y_train[f'{name}_top'] = x
    y_val[f'{name}_top'] = y
    y_test[f'{name}_top'] = z

In [ ]:
from keras.applications import ResNet50
from keras.layers import Input, Flatten, Dense, Concatenate, Activation
from tensorflow.keras.models import Model

def build_resnet_based_model(labels, n_outs, concats, image_size=IMAGE_SIZE):
    def make_top(before_top, n_out, name):
        z = Dense(n_out)(before_top)
        z = Activation('softmax', name=f'{name}_top')(z)
        return z

    def make_branch(res_input, name):
        z = Dense(512, activation='relu')(res_input)
        z = Dense(256, activation='relu')(z)
        z = Dense(128, activation='relu', name=f'{name}_out')(z)
        return z

    def make_concat(inp_main, inp_other, name_main, name_other):
        z = Concatenate(name=f'{name_main}_{name_other}_cat')([inp_main, inp_other])
        z = Dense(128, activation='relu')(z)
        return z
    
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(*image_size, 3))
    base_model.trainable = False
    inputs = Input(shape=(*image_size, 3))
    x = base_model(inputs, training=False)
    x = Flatten()(x)
    x = Dense(1024, activation='relu')(x)
    
    branches = {}
    for label in labels:
        branches[label] = make_branch(x, label)
        
    befores = {}
    for main, other in concats:
        befores[main] = make_concat(branches[main], branches[other], main, other)
        
    outputs = []
    for label in labels:
        if label in befores:
            outputs.append(make_top(befores[label], n_outs[label], label))
        else:
            outputs.append(make_top(branches[label], n_outs[label], label))
            
    
    model = Model(inputs=inputs, outputs=outputs, name='ResnetBased')
    
    return model

In [ ]:
# Dependencies
concats = [
    ('subCategory', 'masterCategory'),
    ('baseColour', 'masterCategory')
]

# Number of outputs
n_outs = {}
for label in labels:
    n_outs[label] = len(np.unique(data_labels[label]))

In [ ]:
from tensorflow.keras.optimizers import Adam

model = build_resnet_based_model(labels, n_outs, concats)

losses = {}
for label in labels:
    losses[f'{label}_top'] = "categorical_crossentropy"
    
metrics = {}
for label in labels:
    metrics[f'{label}_top'] = "accuracy"
    
# Construct class weights dictionary
# class_weight_dict = {label: class_weights[label] for label in df.columns[1:]}


EPOCHS = 25
INIT_LR = 1e-5
BS = 32

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=INIT_LR,
    decay_steps=10000,
    decay_rate=0.9)

opt = Adam(learning_rate=lr_schedule)
model.compile(optimizer=opt, loss=losses, metrics=metrics)

In [ ]:
from tensorflow.keras.utils import plot_model
plot_model(model, show_layer_names=True)

In [ ]:
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(
        X_val, y_val
    ),
    epochs=EPOCHS,
    batch_size=BS,
    callbacks=[early_stopping]
)

In [ ]:
model.save('resnet_based.keras')
pd.DataFrame(history.history).to_csv('history_resnet.csv')

In [ ]:
plts = ['loss', *[f'{label}_top_accuracy' for label in labels]]
plts = [(p, f'val_{p}') for p in plts]

In [ ]:
frame = pd.DataFrame(history.history)
num_subplots = len(frame.columns) // 2
fig, axs = plt.subplots(num_subplots // 3, 3, figsize=(14, 16))

for i, (p, p_v) in enumerate(plts):
    name = p.replace('_top_accuracy', '')
    title = f'{name} Accuracy' if name != 'loss' else 'Loss'
    axs[i // 3, i % 3].set_title(title)
    values_train = frame[p]
    values_valid = frame[p_v]
    axs[i // 3, i % 3].plot(values_train, label='Training')
    axs[i // 3, i % 3].plot(values_valid, label='Validation')
    axs[i // 3, i % 3].legend(loc='lower right')
    axs[i // 3, i % 3].set_xlabel('Epochs')
    axs[i // 3, i % 3].set_ylabel('Accuracy' if name != 'loss' else 'Loss')


plt.show()
    

In [ ]:
for x, y in split:
    print(x.shape, y.shape)

# Export

In [ ]:
# # !zip -r output.zip ./
# from IPython.display import FileLink
# FileLink(r'output.zip')